# Notebook 1: Score Estimation Without Learning (KDE)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FloppingCode/modified-langevin-score-matching/blob/main/notebooks/01_kde.ipynb)

**Goal:** Estimate the score function $\nabla_x \log p(x)$ using Kernel Density Estimation (KDE) -- no neural network training required. We then sample from the estimated distribution using vanilla Langevin dynamics and compare with the analytical score.

**Key idea:** Given data $\{x_i\}_{i=1}^N$, the KDE density is
$$\hat{p}_h(x) = \frac{1}{N} \sum_{i=1}^N \mathcal{N}(x; x_i, h^2 I)$$
and its score is
$$\nabla_x \log \hat{p}_h(x) = \sum_i w_i(x) \frac{x_i - x}{h^2}$$
where $w_i(x) = \text{softmax}\left(-\frac{\|x - x_i\|^2}{2h^2}\right)_i$.

This requires no optimization but scales poorly with dataset size and is sensitive to the bandwidth $h$.

## Setup

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    if os.path.exists("modified-langevin-score-matching"):
        !cd modified-langevin-score-matching && git pull
    else:
        !git clone https://github.com/FloppingCode/modified-langevin-score-matching.git
    sys.path.insert(0, "modified-langevin-score-matching")
else:
    sys.path.insert(0, "..")

In [ ]:
import torch
import matplotlib.pyplot as plt

from dsm import (
    make_dataset, make_dataloader,
    KDEScore,
    vanilla_langevin_dynamics,
    make_8gaussians_analytical_score,
)
from dsm.visualization import plot_samples, animate_sampling, display_animation

## Configuration & Dataset

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Dataset config
DATASET = "8gaussians"
N_SAMPLES = 10_000

# KDE config
BANDWIDTH = 0.1

# Sampling config
N_GENERATED = 2000
N_STEPS = 2000
STEP_SIZE = 1e-3
SAVE_EVERY = 10  # ~200 frames for animation

In [ ]:
dataset = make_dataset(DATASET, n_samples=N_SAMPLES)
data = dataset.tensors[0]
print(f"Dataset shape: {data.shape}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(data[:, 0].numpy(), data[:, 1].numpy(), s=1, alpha=0.5)
ax.set_title(f"{DATASET} dataset ({N_SAMPLES} points)")
ax.set_aspect("equal")
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
plt.show()

## Create KDE and Analytical Score Models

In [ ]:
kde_model = KDEScore(data, bandwidth=BANDWIDTH).to(DEVICE)
analytical_model = make_8gaussians_analytical_score().to(DEVICE)

print(f"KDE score model (bandwidth={BANDWIDTH})")
print(f"Analytical score model (exact Gaussian mixture)")

## Vanilla Langevin Sampling with KDE Score

In [ ]:
kde_samples, kde_traj = vanilla_langevin_dynamics(
    kde_model,
    n_samples=N_GENERATED,
    data_dim=2,
    n_steps=N_STEPS,
    step_size=STEP_SIZE,
    device=DEVICE,
    return_trajectories=True,
    save_every=SAVE_EVERY,
)
print(f"KDE samples: {kde_samples.shape}, trajectory frames: {kde_traj.shape[0]}")

plot_samples(data, kde_samples.cpu(), title="KDE Score Sampling")
plt.show()

## Vanilla Langevin Sampling with Analytical Score

In [ ]:
analytical_samples, analytical_traj = vanilla_langevin_dynamics(
    analytical_model,
    n_samples=N_GENERATED,
    data_dim=2,
    n_steps=N_STEPS,
    step_size=STEP_SIZE,
    device=DEVICE,
    return_trajectories=True,
    save_every=SAVE_EVERY,
)
print(f"Analytical samples: {analytical_samples.shape}, trajectory frames: {analytical_traj.shape[0]}")

plot_samples(data, analytical_samples.cpu(), title="Analytical Score Sampling")
plt.show()

## Side-by-Side Comparison: Real | KDE | Analytical

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

data_np = data.numpy()
kde_np = kde_samples.cpu().numpy()
analytical_np = analytical_samples.cpu().numpy()

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

axes[0].scatter(data_np[:, 0], data_np[:, 1], s=1, alpha=0.5)
axes[0].set_title("Real Data")

axes[1].scatter(kde_np[:, 0], kde_np[:, 1], s=1, alpha=0.5, color="C1")
axes[1].set_title(f"KDE Samples (h={BANDWIDTH})")

axes[2].scatter(analytical_np[:, 0], analytical_np[:, 1], s=1, alpha=0.5, color="C2")
axes[2].set_title("Analytical Samples")

fig.suptitle("Comparison: Real Data vs KDE vs Analytical Score Sampling", fontsize=14)
fig.tight_layout()
plt.show()

## Animation: KDE Sampling Process

In [ ]:
anim_kde = animate_sampling(
    kde_traj,
    real_data=data,
    n_particles=200,
    interval=50,
    title="KDE Langevin Sampling",
    trail_length=10,
)
display_animation(anim_kde)

## Animation: Analytical Score Sampling Process

In [ ]:
anim_analytical = animate_sampling(
    analytical_traj,
    real_data=data,
    n_particles=200,
    interval=50,
    title="Analytical Score Langevin Sampling",
    trail_length=10,
)
display_animation(anim_analytical)

## Bandwidth Sweep

The bandwidth $h$ controls the smoothness of the KDE estimate:
- **Too small** ($h \ll$ data spacing): the score field is very spiky, particles get trapped near individual data points.
- **Too large** ($h \gg$ data spread): the density estimate is over-smoothed, modes blur together.
- **Just right**: captures the multi-modal structure without overfitting to individual points.

In [ ]:
bandwidths = [0.05, 0.2, 0.5]

fig, axes = plt.subplots(1, len(bandwidths), figsize=(5 * len(bandwidths), 5))

for ax, h in zip(axes, bandwidths):
    kde_h = KDEScore(data, bandwidth=h).to(DEVICE)
    samples_h = vanilla_langevin_dynamics(
        kde_h,
        n_samples=N_GENERATED,
        data_dim=2,
        n_steps=N_STEPS,
        step_size=STEP_SIZE,
        device=DEVICE,
        return_trajectories=False,
    )
    samples_np = samples_h.cpu().numpy()
    ax.scatter(samples_np[:, 0], samples_np[:, 1], s=1, alpha=0.5, color="C1")
    ax.set_title(f"h = {h}")
    ax.set_aspect("equal")
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

fig.suptitle("KDE Bandwidth Sweep", fontsize=14)
fig.tight_layout()
plt.show()

**Observations:**
- $h = 0.05$: Sharper modes but may be noisy or miss coverage between clusters.
- $h = 0.2$: Modes start to broaden; still recognizable structure.
- $h = 0.5$: Heavy smoothing collapses the 8-Gaussian structure toward a single blob.

This bandwidth sensitivity is one motivation for learning the score with a neural network instead.

## Summary

- KDE provides a **training-free** score estimate, but its quality is strongly bandwidth-dependent.
- The **analytical score** (exact Gaussian mixture) gives the gold-standard baseline.
- Vanilla Langevin dynamics with the KDE score produces reasonable samples when $h$ is well-chosen.
- Both KDE and analytical use vanilla (non-annealed) Langevin here. The **modified Langevin** correction applies to the annealed setting explored in Notebooks 03 and 04.

**Next:** In Notebook 02, we train a small neural network to learn the score at a single noise level.